In [ ]:
#!pip install requests aiohttp xarray kerchunk pystac_client

In [ ]:
import math
import json

import xarray as xr
import requests
from pystac_client import Client
from fsspec.implementations.reference import ReferenceNotReachable

In [ ]:
response = requests.get("https://staging.eodatahub.org.uk/api/catalogue/stac/catalogs/supported-datasets/catalogs")
response.raise_for_status()

all_catalogs = response.json()

In [ ]:
catalog_urls = {}
for catalog in all_catalogs["catalogs"]:
    for link in catalog["links"]:
        if link["rel"] == "self":
            catalog_urls[catalog["id"]] = link["href"]
            break
    else:
        raise ValueError(f"No catalog link found for catalog {catalog['id']}")

In [ ]:
catalog_urls

In [ ]:
catalog = Client.open(catalog_urls["ceda-stac-catalogue"])

In [ ]:
list(catalog.get_collections())

In [ ]:
renders = {}

for collection in catalog.get_collections():

    if not collection.id.startswith("eocis"):
        continue

    print(collection.id)

    renders[collection.id] = {"renders": {}}
    
    item = next(iter(collection.get_items()))
    if "reference_file" in item.assets:
        asset_url = item.assets["reference_file"].href

        variables_url = f"https://staging.eodatahub.org.uk/titiler/xarray/variables?url={asset_url}"
        response = requests.get(variables_url)
        response.raise_for_status()
        variables = response.json()
        try:
            ds = None
            try:
                ds = xr.open_dataset(asset_url, decode_timedelta=True, engine="kerchunk")
            except ReferenceNotReachable:
                print(f"WARNING: Unable to open {asset_url}")

            for variable in variables:
                
                da = None
                if ds is not None:
                    da = ds[variable]

                if variable == "dtime":
                    continue
                info_url = f"https://staging.eodatahub.org.uk/titiler/xarray/info?url={asset_url}&variable={variable}"
                response = requests.get(info_url)
                if response.status_code == 500:
                    print(f"Info retrieval failed for {variable}")
                    continue
                response.raise_for_status()
                info = response.json()
                print(variable)
                print(info["attrs"]["long_name"])
    
                variable_render = {}
    
                # color map heuristics
                colormap_name = "viridis"
                long_name = info["attrs"]["long_name"]
                if "uncertainty" in long_name.lower() or "uncertainty" in variable.lower():
                    colormap_name = "reds"
                elif any(s in long_name.lower() or s in variable.lower() for s in ["std", "standard deviation", "sdev"]):
                    colormap_name = "blues"              
                elif "sst" in long_name.lower() or "sst" in variable.lower():
                    colormap_name = "plasma"
                elif "lst" in long_name.lower() or "lst" in variable.lower():
                    colormap_name = "cubehelix"
                elif "sea ice" in long_name.lower() or "sea ice" in variable.lower():
                    colormap_name = "blues_r"

                # title optionally with units
                title = long_name.capitalize()
                if "units" in info["attrs"]:
                    units = info["attrs"]["units"]
                    if units.lower() == "kelvin":
                        units = "K"
                    if units.lower() not in ["1"]:
                        title += f" ({units})"
                
                variable_render = {
                    "title": title,
                    "assets": ["reference_file"],
                    "colormap_name": colormap_name,
                    "variable": variable,
                    "reference": True
                }

                # rescale range
                if "actual_range" in info["attrs"]:
                    variable_render["rescale"] = [[
                        math.floor(info["attrs"]["actual_range"][0]),
                        math.ceil(info["attrs"]["actual_range"][1])
                    ]]
                elif da is not None and da.size < 1e9:
                    try:
                        vmin, vmax = map(float, da.quantile((0.05, 0.95)).values)
                        variable_render["rescale"] = [[vmin, vmax]]
                    except ReferenceNotReachable:
                        print(f"WARNING: Unable to load data from {asset_url}. Omitting rescale ranges.")
                else:
                    print(f"WARNING: Unable to determine rescale range for {variable} of size {da.size}.")

                # append to collection of renders
                renders[collection.id]["renders"][variable] = variable_render
        finally:
            if ds is not None:
                ds.close()


In [ ]:
print(json.dumps(renders, indent=2))